# COMP4381: Flight Delays Analysis## Phase 3 - Complete Data Analysis Report**University:** Birzeit University**Course:** COMP4381 - Data Science and Analytics**Instructor:** Dr. Ahmed Sabbah**Date:** May 24, 2026**Status:** Phase 3 Complete

---# 1. IntroductionFlight delays represent one of the most significant challenges in the aviation industry, impacting both operational efficiency and customer satisfaction worldwide. This analysis aims to understand the patterns and factors contributing to flight delays through comprehensive data analysis.## 1.1 Research Questions1. What are the primary factors contributing to flight delays?2. How do delay patterns vary across different time periods and geographic regions?3. Can we build a predictive model for flight delays?## 1.2 Target Variables- **DelayMinutes** (Continuous): Actual delay duration in minutes- **is_delayed** (Binary): Classification variable (1 = delayed >15 min, 0 = on-time ≤15 min)## 1.3 Project Objectives- Integrate flight delay data with airport metadata- Create a clean, representative dataset for analysis- Engineer meaningful features for predictive modeling- Prepare data for Phase 4 machine learning development

---# 2. Data Source Explanation## 2.1 Data Sources### Source 1: Flight Delays Dataset (Kaggle)**File Location:** `C:\Users\user\Downloads\archive\flight_delays.csv`**Dataset Overview:**- Flight operational records containing delay information- Original Size: 1,747,627 flight records**Key Fields:**- **Identifiers:** FlightID, Airline, FlightNumber, AircraftType, TailNumber- **Route Info:** Origin, Destination, Distance (miles)- **Temporal:** ScheduledDeparture, ActualDeparture, ScheduledArrival, ActualArrival- **Target:** DelayMinutes (in minutes)- **Additional:** DelayReason, Cancelled, Diverted### Source 2: Airport Metadata (OurAirports.com)**File Location:** `C:\Users\user\Downloads\airports.csv`**Dataset Overview:**- Global airport reference data- Original Size: 85,462 airport records**Key Fields:**- **Identifier:** iata_code (IATA airport code - merge key)- **Location:** iso_country, municipality, latitude_deg, longitude_deg- **Physical:** elevation_ft- **Classification:** type (airport classification)## 2.2 Data Integration Strategy (LEFT JOIN)**Merge Method:** LEFT JOIN- **Rationale:** Preserves all flight records as the primary dataset- **Implementation:**  1. Origin Merge: Flight Origin → Airport iata_code  2. Destination Merge: Flight Destination → Airport iata_code- **Result:** Each flight enriched with origin and destination airport metadata- **Benefit:** No flight data lost due to missing airport information

---# 3. Data Loading

In [ ]:
import pandas as pdimport numpy as npfrom datetime import datetimefrom pathlib import Pathfrom math import radians, cos, sin, asin, sqrtimport warningswarnings.filterwarnings("ignore")print("Libraries imported successfully")print(f"Execution started: {datetime.now()}")

In [ ]:
# Define file pathsflight_delays_path = r"C:\Users\user\Downloads\archive\flight_delays.csv"airports_path = r"C:\Users\user\Downloads\airports.csv"print("\n" + "="*80)print("FILE PATH VERIFICATION")print("="*80)print(f"\nSource 1 - Flight Delays:")print(f"  Path: {flight_delays_path}")print(f"  Exists: {Path(flight_delays_path).exists()}")print(f"\nSource 2 - Airports:")print(f"  Path: {airports_path}")print(f"  Exists: {Path(airports_path).exists()}")

In [ ]:
# Load the datasetsprint("\n" + "="*80)print("LOADING DATASETS")print("="*80)flight_delays = pd.read_csv(flight_delays_path)airports = pd.read_csv(airports_path)print(f"\nFlight Delays Dataset loaded: {flight_delays.shape}")print(f"  Rows: {len(flight_delays):,}")print(f"  Columns: {flight_delays.shape[1]}")print(f"\nAirports Dataset loaded: {airports.shape}")print(f"  Rows: {len(airports):,}")print(f"  Columns: {airports.shape[1]}")print(f"\nBoth datasets ready for analysis")

---# 4. Dataset Description## 4.1 Flight Delays Dataset - Raw Data Overview

In [ ]:
print("\n" + "="*80)print("FLIGHT DELAYS DATASET - DESCRIPTION")print("="*80)print(f"\nDataset Shape: {flight_delays.shape}")print(f"Memory Usage: {flight_delays.memory_usage(deep=True).sum() / 1024**2:.2f} MB")print(f"\nFirst 5 rows:")display(flight_delays.head())

In [ ]:
print("\nData Types:")print(flight_delays.dtypes)

In [ ]:
print("\nMissing Values:")missing = flight_delays.isnull().sum()if missing.sum() > 0:    print(missing[missing > 0])else:    print("No missing values found")

In [ ]:
print("\nKey Statistics:")print(f"  Total Airlines: {flight_delays['Airline'].nunique()}")print(f"  Total Origin Airports: {flight_delays['Origin'].nunique()}")print(f"  Total Destination Airports: {flight_delays['Destination'].nunique()}")print(f"  Total Unique Flight Numbers: {flight_delays['FlightNumber'].nunique()}")print(f"\nDelay Statistics (in minutes):")print(f"  Mean: {flight_delays['DelayMinutes'].mean():.2f}")print(f"  Median: {flight_delays['DelayMinutes'].median():.2f}")print(f"  Std Dev: {flight_delays['DelayMinutes'].std():.2f}")print(f"  Min: {flight_delays['DelayMinutes'].min():.2f}")print(f"  Max: {flight_delays['DelayMinutes'].max():.2f}")print(f"  25th percentile: {flight_delays['DelayMinutes'].quantile(0.25):.2f}")print(f"  75th percentile: {flight_delays['DelayMinutes'].quantile(0.75):.2f}")

## 4.2 Airports Dataset - Raw Data Overview

In [ ]:
print("\n" + "="*80)print("AIRPORTS DATASET - DESCRIPTION")print("="*80)print(f"\nDataset Shape: {airports.shape}")print(f"Memory Usage: {airports.memory_usage(deep=True).sum() / 1024**2:.2f} MB")print(f"\nFirst 5 rows:")display(airports.head())

In [ ]:
print("\nKey Statistics:")print(f"  Total Airports: {len(airports):,}")print(f"  Countries Represented: {airports['iso_country'].nunique()}")print(f"  Airport Types: {airports['type'].nunique()}")print("\nAirport Type Distribution:")print(airports['type'].value_counts().head(10))

In [ ]:
print("\nAll Countries Represented in the Airports Dataset:")print(f"  Total Countries: {airports['iso_country'].nunique()}")all_countries_sorted = sorted(airports['iso_country'].dropna().unique())countries_str = ", ".join(all_countries_sorted)print(f"\nCountries: {countries_str}")

---# 5. Data Integration (LEFT JOIN)## 5.1 LEFT JOIN ImplementationA **LEFT JOIN** operation was performed to integrate flight delay data with airport metadata. This merge enriches each flight record with geographic and operational information about the origin and destination airports.### JOIN Strategy:**Primary Table:** Flight Delays Dataset (1,747,627 records)**Secondary Table:** Airports Dataset (85,462 records)**Merge Operations:**1. **First JOIN - Origin Airports**   - Left Key: `flight_delays.Origin` (airport code)   - Right Key: `airports.iata_code`   - How: LEFT JOIN   - Result: Adds origin airport metadata (country, coordinates, elevation)2. **Second JOIN - Destination Airports**   - Left Key: `flight_delays.Destination` (airport code)   - Right Key: `airports.iata_code`   - How: LEFT JOIN   - Result: Adds destination airport metadata (country, coordinates, elevation)### Why LEFT JOIN?- **Preserves all flight records**: No flight data is lost- **Enriches with available data**: Adds airport information where available- **Handles missing airports**: If an airport code does not exist in the airport database, the flight record is kept with NULL values for airport fields- **Data quality**: Maintains data integrity and completeness### Join Result:Each flight record now contains:- Original flight information (times, delays, airline, distance)- Origin airport metadata (country, latitude, longitude, elevation)- Destination airport metadata (country, latitude, longitude, elevation)

In [ ]:
print("\n" + "="*80)print("PERFORMING DATA INTEGRATION - LEFT JOIN")print("="*80)print(f"\nBefore JOIN:")print(f"  Flight Delays: {flight_delays.shape}")print(f"  Airports: {airports.shape}")# Prepare airports dataairports_origin = airports[["iata_code", "iso_country", "latitude_deg", "longitude_deg", "elevation_ft", "municipality"]].copy()airports_origin.columns = ["Origin", "origin_country", "origin_latitude", "origin_longitude", "origin_elevation", "origin_city"]airports_dest = airports[["iata_code", "iso_country", "latitude_deg", "longitude_deg", "elevation_ft", "municipality"]].copy()airports_dest.columns = ["Destination", "dest_country", "dest_latitude", "dest_longitude", "dest_elevation", "dest_city"]# Perform LEFT JOINsmerged_data = flight_delays.merge(airports_origin, on="Origin", how="left")merged_data = merged_data.merge(airports_dest, on="Destination", how="left")print(f"\nAfter LEFT JOINs:")print(f"  Merged Dataset: {merged_data.shape}")print(f"  Rows Preserved: {len(merged_data) == len(flight_delays)}")print(f"  New Columns Added: {merged_data.shape[1] - flight_delays.shape[1]}")print(f"\nMissing Values After JOIN:")missing_after = merged_data.isnull().sum()if missing_after.sum() > 0:    print(missing_after[missing_after > 0])else:    print("No missing values")

---# 6. Cities in the Dataset## 6.1 Total Cities OverviewThe dataset contains flight data from **9 major cities** across the United States. All airports and their associated cities are included in this analysis.### The 9 Cities Are:1. **New York** (JFK Airport)2. **Miami** (MIA Airport)3. **Chicago** (ORD Airport)4. **Atlanta** (ATL Airport)5. **Los Angeles** (LAX Airport)6. **Boston** (BOS Airport)7. **San Francisco** (SFO Airport)8. **Seattle** (SEA Airport)9. **Dallas-Fort Worth** (DFW Airport)All cities are located in the United States and represent major aviation hubs with significant flight traffic.

In [ ]:
# Identify all cities in the datasetall_airports = set(flight_delays["Origin"].unique()) | set(flight_delays["Destination"].unique())airport_counts = pd.Series(0, index=all_airports)for airport in flight_delays["Origin"]:    airport_counts[airport] += 1for airport in flight_delays["Destination"]:    airport_counts[airport] += 1# Get cities infocities_list = []for airport_code in sorted(airport_counts.index):    airport_info = airports[airports["iata_code"] == airport_code]    if not airport_info.empty:        city = airport_info.iloc[0].get("municipality", "Unknown")        country = airport_info.iloc[0].get("iso_country", "Unknown")        count = int(airport_counts[airport_code])        pct = (count / airport_counts.sum() * 100)        cities_list.append((airport_code, city, country, count, pct))print("\n" + "="*100)print("ALL CITIES IN THE DATASET")print("="*100)print(f"\nTotal Unique Airports: {len(all_airports)}")print(f"Total Unique Cities: {len(cities_list)}")print(f"\nDetailed Cities List:")for idx, (airport_code, city, country, count, pct) in enumerate(cities_list, 1):    print(f"\n{idx}. {airport_code} - {city}, {country}")    print(f"   Flights: {count:,} ({pct:.1f}%)")

---# 7. Data Sampling Strategy## 7.1 Stratified Sampling ApproachGiven the size of the dataset (1.7M+ rows), stratified random sampling was applied to create a manageable yet representative subset while preserving the class distribution.### Why Stratified Sampling?- **Maintains Class Balance**: Preserves the ratio of on-time vs delayed flights- **Statistical Validity**: Ensures the sample is representative of the population- **Computational Efficiency**: Reduces data size for processing while maintaining data quality- **Reproducibility**: Uses fixed random seed for consistent results### Implementation Details- **Target Sample Size**: 250,000 rows- **Stratification Variable**: `is_delayed` (binary classification: 0=on-time, 1=delayed)- **Random State**: 42 (for reproducibility)- **Sampling Method**: Proportional allocation within each stratum### Expected Outcome- Maintains original class distribution (approximately 63% on-time, 37% delayed)- Final dataset: 250,000 rows with 20 engineered features- Zero missing values after imputation- Ready for machine learning model development

In [ ]:
print("\n" + "="*80)print("STRATIFIED SAMPLING IMPLEMENTATION")print("="*80)# Create binary classification variablemerged_data["is_delayed"] = (merged_data["DelayMinutes"] > 15).astype(int)print(f"\nOriginal Dataset:")print(f"  Total rows: {len(merged_data):,}")print(f"  Class distribution:")class_dist = merged_data["is_delayed"].value_counts(normalize=True)print(f"    On-time (0): {class_dist[0]*100:.1f}%")print(f"    Delayed (1): {class_dist[1]*100:.1f}%")# Perform stratified samplingsample_size = 250000sampled_data = merged_data.groupby("is_delayed", group_keys=False).apply(    lambda x: x.sample(frac=sample_size/len(merged_data), random_state=42))print(f"\nSampled Dataset:")print(f"  Sample size: {len(sampled_data):,}")print(f"  Sample percentage: {len(sampled_data)/len(merged_data)*100:.1f}%")print(f"  Class distribution (preserved):")sample_class_dist = sampled_data["is_delayed"].value_counts(normalize=True)print(f"    On-time (0): {sample_class_dist[0]*100:.1f}%")print(f"    Delayed (1): {sample_class_dist[1]*100:.1f}%")

---# 8. Feature Engineering## 8.1 Feature Engineering OverviewA comprehensive set of 20 features was engineered from the raw data to capture temporal, geographic, and operational patterns that influence flight delays.### Feature Categories:**1. Target Variables (2 features):**- `DelayMinutes` - Continuous delay measurement- `is_delayed` - Binary classification (1 = delayed >15 min, 0 = on-time)**2. Core Identifiers & Route (3 features):**- `FlightNumber` - Flight identifier- `Airline` - Airline operator- `Distance` - Flight distance in miles**3. Geographic Features (4 features):**- `route_distance_km` - Distance in kilometers (Haversine formula)- `elevation_difference` - Difference between origin and destination elevation- `avg_elevation` - Average elevation of origin and destination- `origin_country`, `dest_country` - Geographic location indicators**4. Temporal Features (5 features):**- `departure_month` - Month of departure (1-12)- `departure_hour` - Hour of departure (0-23)- `departure_dayofweek` - Day of week (0-6, Monday=0)- `departure_day` - Day of month (1-31)- `season` - Seasonal classification (Winter, Spring, Summer, Fall)**5. Time Indicators (3 features):**- `is_weekend` - Binary indicator for weekend flights- `is_morning` - Binary indicator for morning departures (6-12)- `is_evening` - Binary indicator for evening departures (18-23)**6. Route & Location (2 features):**- `Origin` - Origin airport code- `Destination` - Destination airport code

In [ ]:
print("\n" + "="*80)print("FEATURE ENGINEERING IMPLEMENTATION")print("="*80)print("\n1. Extracting temporal features...")# Convert to datetimesampled_data["ScheduledDeparture"] = pd.to_datetime(sampled_data["ScheduledDeparture"])# Extract temporal featuressampled_data["departure_month"] = sampled_data["ScheduledDeparture"].dt.monthsampled_data["departure_hour"] = sampled_data["ScheduledDeparture"].dt.hoursampled_data["departure_dayofweek"] = sampled_data["ScheduledDeparture"].dt.dayofweeksampled_data["departure_day"] = sampled_data["ScheduledDeparture"].dt.dayprint("   Done")print("\n2. Creating season feature...")def get_season(month):    if month in [12, 1, 2]:        return "Winter"    elif month in [3, 4, 5]:        return "Spring"    elif month in [6, 7, 8]:        return "Summer"    else:        return "Fall"sampled_data["season"] = sampled_data["departure_month"].apply(get_season)print("   Done")print("\n3. Creating binary time indicators...")sampled_data["is_weekend"] = sampled_data["departure_dayofweek"].isin([5, 6]).astype(int)sampled_data["is_morning"] = sampled_data["departure_hour"].isin(range(6, 13)).astype(int)sampled_data["is_evening"] = sampled_data["departure_hour"].isin(range(18, 24)).astype(int)print("   Done")print("\n4. Creating geographic features...")def haversine(lat1, lon1, lat2, lon2):    try:        lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])        dlon = lon2 - lon1        dlat = lat2 - lat1        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2        c = 2 * asin(sqrt(a))        return c * 6371    except:        return np.nansampled_data["route_distance_km"] = sampled_data.apply(    lambda row: haversine(        row["origin_latitude"], row["origin_longitude"],        row["dest_latitude"], row["dest_longitude"]    ), axis=1)sampled_data["elevation_difference"] = abs(    sampled_data["origin_elevation"].fillna(0) - sampled_data["dest_elevation"].fillna(0))sampled_data["avg_elevation"] = (    sampled_data["origin_elevation"].fillna(0) + sampled_data["dest_elevation"].fillna(0)) / 2print("   Done")print(f"\nFeature Engineering Complete!")

In [ ]:
# Select 20 featuresprint("\n" + "="*80)print("FINAL FEATURE SELECTION - TOP 20 FEATURES")print("="*80)selected_features = [    "DelayMinutes",    "is_delayed",    "FlightNumber",    "Airline",    "Distance",    "Origin",    "Destination",    "route_distance_km",    "elevation_difference",    "avg_elevation",    "departure_month",    "departure_hour",    "departure_dayofweek",    "departure_day",    "season",    "is_weekend",    "is_morning",    "is_evening",    "origin_country",    "dest_country"]selected_features = [f for f in selected_features if f in sampled_data.columns]print(f"\nSelected {len(selected_features)} Features:")for idx, feat in enumerate(selected_features, 1):    dtype = sampled_data[feat].dtype    non_null = sampled_data[feat].notna().sum()    print(f"{idx:2d}. {feat:30s} | Type: {str(dtype):15s} | Non-null: {non_null:,}")# Create final datasetdf_final = sampled_data[selected_features].copy()print(f"\nFinal Dataset Shape: {df_final.shape}")print(f"  Rows: {len(df_final):,}")print(f"  Columns: {df_final.shape[1]}")print(f"  Memory Usage: {df_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")print(f"  Missing Values: {df_final.isnull().sum().sum()}")print(f"\nTarget Variable Distribution:")print(f"  On-time flights: {(df_final['is_delayed'] == 0).sum():,} ({(df_final['is_delayed'] == 0).sum()/len(df_final)*100:.1f}%)")print(f"  Delayed flights: {(df_final['is_delayed'] == 1).sum():,} ({(df_final['is_delayed'] == 1).sum()/len(df_final)*100:.1f}%)")